## 1. Before running the notebook

Start Ollama Model locally with:
```bash
ollama pull llama3.1:8b
```
```bash
ollama serve
```

In [ ]:
import os
import re
import time
import json
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm


## 2. Configs

In [ ]:
# Input/output paths
DATA_PATH = "./complaints.csv"
OUTPUT_PATH = "./synthetic_ollama_responses_1k.csv"
CHECKPOINT_PATH = "./synthetic_ollama_checkpoint.csv"

# Local Ollama settings
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.1:8b"

# Sampling
MAX_SAMPLES = 1000
RANDOM_SEED = 42

# Generation settings
NUM_PREDICT = 150
TEMPERATURE = 0.4
TOP_P = 0.9

# Runtime controls
REQUEST_TIMEOUT = 180
SAVE_EVERY = 25
RESUME_FROM_CHECKPOINT = True

print("Configured model:", OLLAMA_MODEL)
print("Output path:", OUTPUT_PATH)


Configured model: llama3.1:8b
Output path: ./synthetic_ollama_responses_1k.csv


## 3. Check Ollama connection

In [3]:
def check_ollama() -> None:
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=10)
        r.raise_for_status()
        models = [m.get("name", "") for m in r.json().get("models", [])]
        print("Ollama is running.")
        print("Installed models:")
        for m in models:
            print(" -", m)
        if OLLAMA_MODEL not in models:
            print(f"\nWarning: {OLLAMA_MODEL} not found. Run: ollama pull {OLLAMA_MODEL}")
    except Exception as e:
        raise RuntimeError(
            "Could not connect to Ollama. Make sure 'ollama serve' is running locally."
        ) from e

check_ollama()


Ollama is running.
Installed models:
 - llama3.1:8b
 - qwen2.5:latest
 - qwen2.5:7b-instruct
 - llama3.2:3b


## 4. Load and preprocess complaints

In [ ]:
def load_complaints(path: str) -> pd.DataFrame:
    path = Path(path)

    df = pd.read_csv(path)

    df = df.dropna(subset=["narrative"]).copy()
    df["narrative"] = (
        df["narrative"]
        .astype(str)
        .str.strip()
        .str.replace(r"\bx{2,}\b", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Length filtering
    df = df[df["narrative"].str.len() > 50].copy()
    df["word_count"] = df["narrative"].str.split().str.len()
    df = df[(df["word_count"] >= 25) & (df["word_count"] <= 300)].copy()
    df = df.drop_duplicates(subset=["narrative"]).copy()

    keep_cols = ["narrative", "product", "word_count"]
    df = df[keep_cols].copy()

    n = min(MAX_SAMPLES, len(df))
    df = df.sample(n=n, random_state=RANDOM_SEED).reset_index(drop=True)
    df = df.sort_values("word_count").reset_index(drop=True)

    return df

df = load_complaints(DATA_PATH)
print(df.shape)
df.head()


(1000, 3)


,narrative,product,word_count
0,month called creditor request proof balance co...,debt_collection,25
1,regularly transfer money bank account tried to...,retail_banking,25
2,recently sent lvnv funding llc letter disputin...,debt_collection,25
3,account open yet report closed stated falsely ...,credit_reporting,25
4,account account account belong mean person acc...,credit_reporting,25


## 5. Prompt and validation functions

In [ ]:
SYSTEM_PROMPT = (
    "You are a banking customer support specialist responding to a logged complaint. "
    "Write ONE paragraph (90-140 words). "

    "Strict rules: "
    "1. Do NOT start with phrases like 'Dear valued customer' or 'Thank you for reaching out'. "
    "2. Use ONLY information from the complaint. Do NOT invent company names, institutions, or details. "
    "2a. Do NOT assume the issue involves your own company unless explicitly stated. "
    "3. Clearly acknowledge the customer's issue using specific details from the complaint. "
    "4. Explain what the support team will review or investigate internally. "
    "5. Do NOT ask the customer for any information. "
    "6. Do NOT include phone numbers, email addresses, or contact details. "
    "7. Do NOT use bullet points, numbered lists, or markdown formatting. "

    "Write in clear, natural, professional prose."
)

def build_prompt(row: dict) -> str:
    product = str(row.get("product", "")).strip()
    complaint = str(row.get("narrative", "")).strip()

    return f"""{SYSTEM_PROMPT}

Product category: {product}

Customer complaint:
{complaint}

Response:"""

def clean_response(text: str) -> str:
    text = str(text or "").strip()

    # Remove markdown formatting while keeping words
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r'[`*_#>"]', "", text)

    # Remove common model preambles
    text = re.sub(r"^(response:|assistant:)", "", text, flags=re.IGNORECASE).strip()

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Trim to last complete sentence if possible
    last_end = max(text.rfind("."), text.rfind("!"), text.rfind("?"))
    if last_end > 80:
        text = text[:last_end + 1].strip()

    return text

def is_valid_response(text: str) -> bool:
    text = str(text or "").strip()
    lower = text.lower()
    words = text.split()

    if len(words) < 45 or len(words) > 190:
        return False

    bad_patterns = [
        "dear valued customer",
        "thank you for reaching out",
        "thank you for your feedback",
        "account number",
        "phone number",
        "email address",
        "support@example",
        "555-",
        "could you please provide",
        "please provide",
        "please share",
        "kindly provide",
        "contact us at",
        "we've reviewed",
        "we have reviewed",
        "i have reviewed",
        "after reviewing",
        "our company",
        "our representative",
        "company name",
        "creditor's name",
        "placeholder",
        "[",
        "]",
        "unable to validate the debt",
    ]

    if any(p in lower for p in bad_patterns):
        return False

    if (
        "**" in text
        or re.search(r"\s*[-*]\s+", text)
        or re.search(r"\s*\d+\.\s+", text)
    ):
        return False

    if not text or text[-1] not in {".", "!", "?", '"', "'"}:
        return False

    return True


## 6. Test one sample

In [6]:
def generate_ollama(row: dict) -> str:
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": build_prompt(row),
        "stream": False,
        "options": {
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "num_predict": NUM_PREDICT,
        },
    }

    r = requests.post(OLLAMA_URL, json=payload, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    data = r.json()
    return clean_response(data.get("response", ""))

sample_row = df.iloc[0].to_dict()
sample_text = generate_ollama(sample_row)

print("Complaint:", sample_row["narrative"])
print("Generated response:", sample_text)
print("Valid:", is_valid_response(sample_text))
print("Word count:", len(sample_text.split()))


Complaint: month called creditor request proof balance contract would make liable debt well balance asked validate debt claimed owed regard information requested unable provide documentation requested
Generated response: We've reviewed your complaint regarding the conversation you had with a representative from [Creditor's Name] last month. You mentioned that they requested proof of your balance and contract, claiming that you're liable for the debt. However, you were unable to provide the documentation as you disputed the information provided by the creditor. Our team will investigate this matter further and review the communication between you and the creditor to ensure that all relevant details are accurate and up-to-date. We'll also verify the validity of the debt claimed and confirm whether the creditor has followed the necessary procedures for debt collection.
Valid: False
Word count: 100


## 7. Generate responses


In [7]:
def load_checkpoint_if_available(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str], int]:
    if RESUME_FROM_CHECKPOINT and Path(CHECKPOINT_PATH).exists():
        ckpt = pd.read_csv(CHECKPOINT_PATH)
        if "response" in ckpt.columns and len(ckpt) > 0:
            done = len(ckpt)
            responses = ckpt["response"].fillna("").astype(str).tolist()
            print(f"Resuming from checkpoint: {done}/{len(df)} rows already generated")
            return df, responses, done

    return df, [], 0

def save_checkpoint(df: pd.DataFrame, responses: list[str]) -> None:
    partial = df.iloc[:len(responses)].copy()
    partial["response"] = responses
    partial.to_csv(CHECKPOINT_PATH, index=False)

responses_df, responses, start_idx = load_checkpoint_if_available(df)

for i in tqdm(range(start_idx, len(df)), desc="Generating local Ollama responses"):
    row = df.iloc[i].to_dict()

    try:
        response = generate_ollama(row)
    except Exception as e:
        print(f"Row {i} failed: {str(e)[:120]}")
        response = ""

    responses.append(response)

    if (i + 1) % SAVE_EVERY == 0:
        save_checkpoint(df, responses)

# Final checkpoint
save_checkpoint(df, responses)
print(f"Generated {len(responses)} responses")


Resuming from checkpoint: 25/1000 rows already generated


Generating local Ollama responses: 100%|██████████| 975/975 [1:30:09<00:00,  5.55s/it]

Generated 1000 responses


## 8. Filter and save final CSV

In [8]:
synth_df = df.iloc[:len(responses)].copy()
synth_df["response"] = responses

before = len(synth_df)
synth_df = synth_df[synth_df["response"].fillna("").astype(str).str.strip().str.len() > 0].copy()
print(f"Removed {before - len(synth_df)} empty responses")

valid_mask = synth_df["response"].apply(is_valid_response)
print(f"Removing {(~valid_mask).sum()} responses that failed validation")
synth_df = synth_df[valid_mask].copy()

synth_df = synth_df.drop(columns=["word_count"], errors="ignore").reset_index(drop=True)
synth_df.to_csv(OUTPUT_PATH, index=False)

print("Final clean dataset:", synth_df.shape)
print("Saved to:", OUTPUT_PATH)
synth_df[["narrative", "product", "response"]].head()


Removed 0 empty responses
Removing 350 responses that failed validation
Final clean dataset: (650, 3)
Saved to: ./synthetic_ollama_responses_1k.csv


,narrative,product,response
0,recently sent lvnv funding llc letter disputin...,debt_collection,We have received your letter disputing the all...
1,account open yet report closed stated falsely ...,credit_reporting,We are reviewing your complaint regarding the ...
2,account account account belong mean person acc...,credit_reporting,We understand that you're concerned about the ...
3,paid charged account full kohl impression woul...,debt_collection,We understand that you're concerned about the ...
4,account open yet report closed stated falsely ...,credit_reporting,We have received your complaint regarding the ...
